# 02 — Spans

Spans let you extract the **per-layer mean hidden state** of a specific sub-sequence of the input — a subject, a cue phrase, a relation, a named entity — and store it as a dedicated array.

This notebook covers:
- `TextSpan` — robust substring-based spans (recommended)
- `SpanSpec` — explicit token-index spans
- The `label` parameter and list form
- The `occurrence` parameter for repeated substrings
- How resolved spans are stored in `InternalsRecord`
- Practical analysis: comparing subject vs. object representations across layers

In [ ]:
import internals_extraction
from internals_extraction import (
    InternalsDataset, InternalsInstance,
    TextSpan, SpanSpec, SpanResolutionError,
)
from transformers import AutoModelForCausalLM, AutoTokenizer
import numpy as np

MODEL     = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model     = AutoModelForCausalLM.from_pretrained(MODEL)
model.eval()
print("Ready")

## 1. Why TextSpan?

Token indices are fragile: adding a BOS token, switching to a different tokenizer version, or changing whitespace handling can silently shift every index.

`TextSpan` defines a span by the **original text substring** and resolves it to token indices at inference time using `return_offsets_mapping=True`.

In [ ]:
text = "The cat sat on the mat."

# Show what the tokenizer produces
enc    = tokenizer(text)
tokens = tokenizer.convert_ids_to_tokens(enc["input_ids"])
print("Tokens:", list(enumerate(tokens)))

In [ ]:
# TextSpan: defined by substring — stable regardless of tokenizer internals
instance = InternalsInstance(
    text=text,
    properties={"id": 0},
    spans={
        "subject":   TextSpan("The cat"),
        "predicate": TextSpan("sat on the mat"),
    },
)

dataset = InternalsDataset([instance])
records = dataset.run(model, tokenizer, generate_kwargs={"max_new_tokens": 1})
rec     = records[0]

print("Resolved spans (token indices):")
for name, span in rec.resolved_spans.items():
    toks = tokens[span.start:span.end]
    print(f"  {name:>12}: tokens {span.start}–{span.end}  → {toks}")

print("\nSpan mean hidden-state shapes:")
for name, arr in rec.span_hidden_states_mean.items():
    print(f"  {name:>12}: {arr.shape}   (num_layers, hidden)")

## 2. TextSpan with `label` — list form

When building instances programmatically it is convenient to have each span carry its own name. Pass a **list** of `TextSpan` objects with `label` set and they will be converted to a dict automatically.

In [ ]:
text = "Albert Einstein was born in Ulm, Germany."

instance = InternalsInstance(
    text=text,
    properties={"entity": "Albert Einstein", "task": "biography"},
    spans=[
        TextSpan("Albert Einstein", label="entity"),
        TextSpan("was born in",     label="relation"),
        TextSpan("Ulm, Germany",    label="location"),
    ],
)

print("Span keys after normalisation:", list(instance.spans.keys()))

records = InternalsDataset([instance]).run(
    model, tokenizer, generate_kwargs={"max_new_tokens": 1}
)
rec = records[0]

print("\nResolved spans:")
for name, span in rec.resolved_spans.items():
    print(f"  {name:>10}: tokens {span.start}–{span.end}")

## 3. The `occurrence` parameter

When the same substring appears multiple times, `occurrence=k` selects the (k+1)-th match.

In [ ]:
text = "the cat chased the dog because the cat was hungry"

enc    = tokenizer(text)
tokens = tokenizer.convert_ids_to_tokens(enc["input_ids"])
print("Tokens:", list(enumerate(tokens)))
print()

instance = InternalsInstance(
    text=text,
    spans=[
        TextSpan("the cat", label="subject_1",  occurrence=0),  # first
        TextSpan("the dog", label="object",      occurrence=0),
        TextSpan("the cat", label="subject_2",  occurrence=1),  # second
    ],
)

records = InternalsDataset([instance]).run(
    model, tokenizer, generate_kwargs={"max_new_tokens": 1}
)
rec = records[0]

print("Resolved spans:")
for name, span in rec.resolved_spans.items():
    toks = tokens[span.start:span.end]
    print(f"  {name:>12}: tokens {span.start}–{span.end}  → {toks}")

In [ ]:
# Out-of-range occurrence raises a clear error
try:
    bad_instance = InternalsInstance(
        text=text,
        spans={"bad": TextSpan("the cat", occurrence=5)},
    )
    InternalsDataset([bad_instance]).run(model, tokenizer)
except SpanResolutionError as e:
    print(f"SpanResolutionError: {e}")

## 4. SpanSpec — explicit token indices

Use when you already know the token positions (e.g. from a pre-tokenised dataset with BIO labels). Supports negative indices (Python-style).

In [ ]:
text = "[CLS] The cat sat on the mat . [SEP]"

instance = InternalsInstance(
    text=text,
    spans={
        "cls":      SpanSpec(0, 1),    # token 0 only
        "content":  SpanSpec(1, -1),   # everything except first and last
        "sep":      SpanSpec(-1, 0),   # would be empty → zero vector
    },
)

records = InternalsDataset([instance]).run(
    model, tokenizer, generate_kwargs={"max_new_tokens": 1}
)
rec = records[0]

for name, arr in rec.span_hidden_states_mean.items():
    is_zero = np.allclose(arr, 0)
    print(f"  {name:>10}: shape={arr.shape}  all-zero={is_zero}")

## 5. Analysis: subject vs. object representations across layers

Compare the cosine similarity between entity and location hidden-state means at each layer depth.

In [ ]:
sentences = [
    ("Marie Curie was born in Warsaw, Poland.",
     "Marie Curie", "Warsaw, Poland"),
    ("Albert Einstein was born in Ulm, Germany.",
     "Albert Einstein", "Ulm, Germany"),
    ("Isaac Newton was born in Woolsthorpe, England.",
     "Isaac Newton", "Woolsthorpe, England"),
]

instances = [
    InternalsInstance(
        text=text,
        properties={"person": person, "location": location},
        spans=[
            TextSpan(person,   label="entity"),
            TextSpan(location, label="location"),
        ],
    )
    for text, person, location in sentences
]

records = InternalsDataset(instances).run(
    model, tokenizer, generate_kwargs={"max_new_tokens": 1}
)

print("Cosine similarity between entity and location representations per layer:\n")
for rec in records:
    entity_hs   = rec.span_hidden_states_mean["entity"]    # (L, H)
    location_hs = rec.span_hidden_states_mean["location"]  # (L, H)
    print(f"  {rec.properties['person']} / {rec.properties['location']}")
    for layer in range(entity_hs.shape[0]):
        a, b = entity_hs[layer], location_hs[layer]
        cos  = np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9)
        bar  = "█" * int(abs(cos) * 20)
        print(f"    Layer {layer:2d}  cos={cos:+.4f}  {bar}")
    print()

## 6. Mixing span types in one instance

In [ ]:
instance = InternalsInstance(
    text="The quick brown fox jumps over the lazy dog.",
    spans={
        # TextSpan — substring
        "subject":  TextSpan("The quick brown fox"),
        "verb":     TextSpan("jumps"),
        "object":   TextSpan("the lazy dog"),
        # SpanSpec — token indices
        "first_3": SpanSpec(0, 3),
        # string shorthand (auto-converted to TextSpan)
        "over":    "over",
    },
)

records = InternalsDataset([instance]).run(
    model, tokenizer, generate_kwargs={"max_new_tokens": 1}
)
rec = records[0]

print("Span shapes:")
for name, arr in rec.span_hidden_states_mean.items():
    print(f"  {name:>10}: {arr.shape}")

print("\nResolved token indices:")
for name, span in rec.resolved_spans.items():
    print(f"  {name:>10}: [{span.start}, {span.end})")